# LiH Dissociation Curve — Scaling to a 12-Qubit Molecule

LiH (Lithium Hydride) is a 12-qubit problem (6 spatial orbitals) using STO-3G basis.
It dissociates into neutral Li and H atoms, introducing more orbital complexity
than H₂ and serving as a realistic test case for scaling VQE methods.

We compute the dissociation curve using three methods:
- Exact classical diagonalization
- Standard UCCSD-VQE
- ADAPT-VQE

## Step 1 — Imports

In [2]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
np.set_printoptions(precision=6, suppress=True)

from qiskit_algorithms import VQE, AdaptVQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD
from qiskit_nature.second_q.circuit.library.initial_states import HartreeFock
# from qiskit.primitives import Estimator

print("All imports OK")

All imports OK


## Step 2 — Dissociation Curve Sweep

Bond lengths from 1.0Å to 4.0Å (20 points).
Li at origin, H moves along x-axis.

In [11]:
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit import transpile
from qiskit_aer import AerSimulator

# Create a simulator object to use as a target for transpilation
sim = AerSimulator()

# Transpile the ansatz to the simulator's basis gates
# We use optimization_level=1 to keep it fast




bond_lengths = np.linspace(1.0, 4.0, 3)
exact_energies = []
uccsd_energies = []
adapt_energies = []
# Replace:
# With:
estimator = AerEstimator()


print(f"Running LiH dissociation curve for {len(bond_lengths)} bond lengths...")
print(f"Range: {bond_lengths[0]:.2f}Å to {bond_lengths[-1]:.2f}Å")
print("-" * 65)

for i, dist in enumerate(bond_lengths):
    mol = f"Li 0.0 0.0 0.0\nH {dist:.3f} 0.0 0.0\n"
    driver = PySCFDriver(atom=mol, basis="sto3g")
    problem = driver.run()

    num_spatial = problem.num_spatial_orbitals
    num_particles = problem.num_particles
    nuclear_repulsion = problem.nuclear_repulsion_energy

    hamiltonian = problem.hamiltonian
    second_q_op = hamiltonian.second_q_op()
    mapper = JordanWignerMapper()
    qubit_op = mapper.map(second_q_op)
    exact_matrix = qubit_op.to_matrix()
    exact_eig = np.linalg.eigh(exact_matrix).eigenvalues[0]
    exact_energies.append(exact_eig + nuclear_repulsion)

    initial_state = HartreeFock(
        num_spatial_orbitals=num_spatial,
        num_particles=num_particles,
        qubit_mapper=mapper,
    )

    ansatz = UCCSD(
        num_spatial_orbitals=num_spatial,
        num_particles=num_particles,
        qubit_mapper=mapper,
        initial_state=initial_state,
    )

    transpiled_ansatz = transpile(ansatz, sim, optimization_level=1)

    vqe = VQE(
        estimator=estimator,
        ansatz=transpiled_ansatz, # <--- Use the transpiled version
        optimizer=COBYLA(maxiter=50),
    )
    result_uccsd = vqe.compute_minimum_eigenvalue(qubit_op)
    uccsd_energies.append(result_uccsd.eigenvalue.real + nuclear_repulsion)

    pool_ops = [mapper.map(op) for op in ansatz.excitation_ops()]
    # In recent qiskit-algorithms, AdaptVQE requires a VQE solver instance
    transpiled_init = transpile(initial_state, sim) 
    solver = VQE(estimator=estimator, ansatz=transpiled_init, optimizer=COBYLA(maxiter=300))
    adapt_vqe = AdaptVQE(
        solver=solver,
        operators=pool_ops,
        gradient_threshold=1e-6,
    )
    result_adapt = adapt_vqe.compute_minimum_eigenvalue(qubit_op)
    adapt_energies.append(result_adapt.eigenvalue.real + nuclear_repulsion)

    print(f"  [{i+1:2d}/{len(bond_lengths)}] d={dist:.3f}Å | "
          f"exact={exact_energies[-1]:.6f} | uccsd={uccsd_energies[-1]:.6f} | "
          f"adapt={adapt_energies[-1]:.6f}")

print("-" * 65)
print("Done!")

Running LiH dissociation curve for 3 bond lengths...
Range: 1.00Å to 4.00Å
-----------------------------------------------------------------


AlgorithmError: 'All gradients have been evaluated to lie below the convergence threshold during the first iteration of the algorithm. Try to either tighten the convergence threshold or pick a different ansatz.'

## Step 3 — Dissociation Curve Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(bond_lengths, exact_energies, 'k-', linewidth=2.5,
        label='Exact (Classical)', zorder=5)
ax.plot(bond_lengths, uccsd_energies, 'g-s', markersize=5, linewidth=1.5,
        label='UCCSD-VQE', zorder=4)
ax.plot(bond_lengths, adapt_energies, 'b-o', markersize=5, linewidth=1.5,
        label='ADAPT-VQE', zorder=4)

eq_idx = np.argmin(np.abs(bond_lengths - 1.6))
ax.scatter([bond_lengths[eq_idx]], [exact_energies[eq_idx]],
           color='red', s=120, zorder=6, marker='*',
           label=f'Equilibrium ({bond_lengths[eq_idx]:.3f}Å)')

ax.set_xlabel('Bond Length (Angstrom)', fontsize=13)
ax.set_ylabel('Ground State Energy (Hartree)', fontsize=13)
ax.set_title('LiH Dissociation Curve\nExact vs UCCSD-VQE vs ADAPT-VQE', fontsize=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

textstr = (f'Qubits: {qubit_op.num_qubits}\n'
            f'Spatial orbitals: {num_spatial}\n'
            f'Electrons: {num_particles}')
ax.text(0.03, 0.03, textstr, transform=ax.transAxes,
         fontsize=10, verticalalignment='bottom',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('vqe_h2/lih_dissociation_curve.png', dpi=150)
plt.show()

## Step 4 — Error Analysis

In [ ]:
uccsd_errors = [abs(e - ex) for e, ex in zip(uccsd_energies, exact_energies)]
adapt_errors = [abs(e - ex) for e, ex in zip(adapt_energies, exact_energies)]

fig, ax = plt.subplots(figsize=(12, 6))

ax.semilogy(bond_lengths, uccsd_errors, 'g-s', markersize=5, linewidth=1.5,
            label='UCCSD-VQE error')
ax.semilogy(bond_lengths, adapt_errors, 'b-o', markersize=5, linewidth=1.5,
            label='ADAPT-VQE error')
ax.axhline(y=0.0016, color='red', linestyle='--', linewidth=1.5,
           label='Chemical accuracy (1.6 mHa)')

ax.set_xlabel('Bond Length (Angstrom)', fontsize=13)
ax.set_ylabel('|Error| vs Exact (Hartree, log scale)', fontsize=13)
ax.set_title('LiH VQE Error vs Bond Length: UCCSD vs ADAPT', fontsize=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('vqe_h2/lih_dissociation_error.png', dpi=150)
plt.show()

print(f"Max UCCSD error: {max(uccsd_errors):.6f} Ha")
print(f"Max ADAPT error: {max(adapt_errors):.6f} Ha")
print(f"Mean UCCSD error: {np.mean(uccsd_errors):.6f} Ha")
print(f"Mean ADAPT error: {np.mean(adapt_errors):.6f} Ha")

## Step 5 — Summary

In [ ]:
print("=" * 70)
print(f"{'LiH DISSOCIATION CURVE SUMMARY':^70}")
print("=" * 70)
print(f"Bond length range: {bond_lengths[0]:.2f}Å to {bond_lengths[-1]:.2f}Å ({len(bond_lengths)} points)")
print(f"Equilibrium energy (exact): {exact_energies[eq_idx]:.12f} Ha")
print("-" * 70)
print(f"{'Metric':<35} {'UCCSD-VQE':<18} {'ADAPT-VQE':<18}")
print("-" * 70)
print(f"{'Max error (Ha)':<35} {max(uccsd_errors):<18.8f} {max(adapt_errors):<18.8f}")
print(f"{'Mean error (Ha)':<35} {np.mean(uccsd_errors):<18.8f} {np.mean(adapt_errors):<18.8f}")
print(f"{'Points within chemical accuracy':<35} "
      f"{sum(e < 0.0016 for e in uccsd_errors):<18} {sum(e < 0.0016 for e in adapt_errors):<18}")
print("=" * 70)